In [ ]:
from pathlib import Path
import os
from os import listdir as ls
from scipy.stats import gaussian_kde
import numpy as np
import arviz as az
import pandas as pd
import pycountry
import pycountry_convert as pc
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
import seaborn as sns
from IPython.display import display

from emu_renewal.constants import OUTPUTS_PATH, MOB_SOURCE_COLOURS
from emu_renewal.plotting import get_standard_subplot
from emu_renewal.utils import get_countries_by_continent, get_cont_of_country, sort_countries_by_name, split_countries_two_groups

set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "47213985"
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)
split_countries_by_cont = split_countries_two_groups(countries_by_cont, 15)

In [ ]:
for cont, countries in split_countries_by_cont.items():
    for c, iso3 in enumerate(countries[:1]):
        analyses = {d.name: Path(d.path) for d in os.scandir(job_path / iso3) if d.is_dir()}
        no_mob_path = analyses["no_mob"]
        idatas = {a: az.from_netcdf(p / "idata_filtered.nc") for a, p in analyses.items()}

In [ ]:
idata = idatas["no_mob"]
log_likelihood = idata.log_likelihood
idata.log_likelihood["total_log_likelihood"] = sum([log_likelihood[var] for var in log_likelihood.data_vars])
waic_result = az.waic(idata, var_name="total_log_likelihood")